# 03 — Sub-builders: composing HTML with SVG (and HTML again)

How to switch dialect mid-document with `@subbuilder`. Inside an
`HtmlBuilderHandler` you can call `body.svg()` and continue with the
SVG grammar; inside SVG you can re-enter HTML with `svg.html(...).div(...)`
and the framework wraps the HTML subtree in `<foreignObject xmlns=...>`
at render time. Each section shows the source, the rendered HTML
markup, and the visual result.

**Topics:**

1. HTML hosting SVG — a shape inline.
2. A composable status badge — icon + label in one flow.
3. SVG primitives in one figure — rect, circle, ellipse, line, polygon, text.
4. HTML overlay inside SVG via `svg.html()` — wrap-tag mechanism in action.
5. Dashboard card — three-switch composition (HTML → SVG → HTML).
6. Inspection — how the framework realises sub-builders.

Prerequisites: notebooks `01_introduction`, `02_inline_styling`.

## 1. HTML hosting SVG

`body.svg(...)` opens the SVG dialect. From that point on, descendants
are validated against the SVG grammar: `rect`, `circle`, `path` are
available; `div`, `p` are not (they belong to HTML).

The framework attaches an `SvgBuilder` instance to the `<svg>` node's
`_builder` slot, and dispatch follows it. The HTML host renderer hands
the subtree off to the SVG renderer at output time — so `<rect />`
comes out with the SVG convention (space before the slash), while
the surrounding `<body>` stays HTML5-style.

In [1]:
from IPython.display import HTML

from genro_builders.contrib.html import HtmlBuilderHandler


class Section1(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.h2('Two shapes inline')
        s = body.svg(width=200, height=80, viewBox='0 0 200 80')
        s.circle(cx=40, cy=40, r=30, fill='#e74c3c')
        s.rect(x=90, y=20, width=90, height=40, fill='#3498db')


p1 = Section1(); p1.create(); p1.build()
print(p1.render(pretty=True))

<body>
  <h2>Two shapes inline</h2>
<svg width="200" height="80" viewBox="0 0 200 80"><circle cx="40" cy="40" r="30" fill="#e74c3c" /><rect x="90" y="20" width="90" height="40" fill="#3498db" /></svg></body>



In [2]:
HTML(p1.render())

## 2. A composable status badge

An SVG icon next to an HTML label, sharing the same parent flexbox
container. The icon is built from primitive SVG elements (a circle
background + a checkmark drawn as a `<path>`). The text label is a
plain HTML `<span>`. Both live inside the same flex row because the
flex container is HTML and the SVG is just a child of it.

In [3]:
class Section2(HtmlBuilderHandler):
    def main(self, root):
        body = root.body(display='flex', align_items='center', gap='12px')
        s = body.svg(width=32, height=32, viewBox='0 0 32 32')
        s.circle(cx=16, cy=16, r=15, fill='#27ae60')
        s.path(
            d='M9 16 l5 5 l9 -10',
            stroke='white', stroke_width=3, fill='none',
            stroke_linecap='round', stroke_linejoin='round',
        )
        body.span('Online', color='#27ae60', font_weight='600')


p2 = Section2(); p2.create(); p2.build()
print(p2.render(pretty=True))

<body style="display: flex; align-items: center; gap: 12px">
<svg width="32" height="32" viewBox="0 0 32 32"><circle cx="16" cy="16" r="15" fill="#27ae60" /><path d="M9 16 l5 5 l9 -10" stroke="white" stroke-width="3" fill="none" stroke-linecap="round" stroke-linejoin="round" /></svg>  <span style="color: #27ae60; font-weight: 600">Online</span>
</body>



In [4]:
HTML(p2.render())

## 3. SVG primitives in one figure

Every shape used here belongs to the SVG grammar: `rect`, `circle`,
`ellipse`, `line`, `polygon`, `text`. The host HTML page does not
need to know about any of them — the dispatch through `_builder`
makes sure each call lands on the right grammar.

In [5]:
class Section3(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.h2('SVG primitives in one figure')
        s = body.svg(width=360, height=160, viewBox='0 0 360 160')
        s.rect(x=0, y=0, width=360, height=160, fill='#ecf0f1')
        s.circle(cx=60, cy=40, r=24, fill='#1abc9c')
        s.circle(cx=130, cy=40, r=24, fill='#3498db')
        s.circle(cx=200, cy=40, r=24, fill='#9b59b6')
        s.ellipse(cx=300, cy=40, rx=40, ry=20, fill='#e67e22')
        s.line(x1=20, y1=90, x2=340, y2=90, stroke='#34495e', stroke_width=2)
        s.polygon(points='60,150 100,110 140,150', fill='#e74c3c')
        s.text('inline SVG', x=170, y=140, font_size=14, fill='#2c3e50')


p3 = Section3(); p3.create(); p3.build()
print(p3.render(pretty=True))

<body>
  <h2>SVG primitives in one figure</h2>
<svg width="360" height="160" viewBox="0 0 360 160"><rect x="0" y="0" width="360" height="160" fill="#ecf0f1" /><circle cx="60" cy="40" r="24" fill="#1abc9c" /><circle cx="130" cy="40" r="24" fill="#3498db" /><circle cx="200" cy="40" r="24" fill="#9b59b6" /><ellipse cx="300" cy="40" rx="40" ry="20" fill="#e67e22" /><line x1="20" y1="90" x2="340" y2="90" stroke="#34495e" stroke-width="2" /><polygon points="60,150 100,110 140,150" fill="#e74c3c" /><text x="170" y="140" font-size="14" fill="#2c3e50">inline SVG</text></svg></body>



In [6]:
HTML(p3.render())

## 4. HTML overlay inside SVG via `svg.html(...)`

Inside an SVG drawing you can place rich HTML — paragraphs, lists,
styled text — through the `svg.html()` re-entry. The source bag
keeps the readable tag `html`; the framework wraps it in a
`<foreignObject xmlns="http://www.w3.org/1999/xhtml">` envelope
at render time, with the W3C-required namespace already on the
wrap tag. User attributes (`x`, `y`, `width`, `height`) on the
`html` call land on the foreignObject — that is where SVG needs
them to position the embedded HTML block.

In [7]:
class Section4(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.h2('HTML overlay inside SVG')
        s = body.svg(width=360, height=180, viewBox='0 0 360 180')
        s.rect(x=0, y=0, width=360, height=180, rx=12, ry=12, fill='#2c3e50')
        s.circle(cx=40, cy=40, r=20, fill='#f1c40f')
        overlay = s.html(x=80, y=20, width=260, height=140)
        card = overlay.div(
            color='white', font_family='sans-serif', padding='8px',
        )
        card.div('Mixed content', font_weight='700', font_size='16px')
        card.p(
            'This block is HTML rendered inside an SVG, ideal for '
            'long-form copy that SVG <text> cannot wrap natively.',
            font_size='13px', margin='6px 0 0 0',
        )


p4 = Section4(); p4.create(); p4.build()
print(p4.render(pretty=True))

<body>
  <h2>HTML overlay inside SVG</h2>
<svg width="360" height="180" viewBox="0 0 360 180"><rect x="0" y="0" width="360" height="180" rx="12" ry="12" fill="#2c3e50" /><circle cx="40" cy="40" r="20" fill="#f1c40f" /><foreignObject xmlns="http://www.w3.org/1999/xhtml" x="80" y="20" width="260" height="140"><div style="color: white; font-family: sans-serif; padding: 8px"><div style="font-weight: 700; font-size: 16px">Mixed content</div><p style="font-size: 13px; margin: 6px 0 0 0">This block is HTML rendered inside an SVG, ideal for long-form copy that SVG &lt;text&gt; cannot wrap natively.</p></div></foreignObject></svg></body>



In [8]:
HTML(p4.render())

## 5. Dashboard card — three-switch composition

A miniature dashboard tile: HTML wrapper, SVG bar chart in the
middle, and an HTML caption *inside* the SVG via `svg.html()`.
Three dialect switches in one coherent piece of markup, with
each layer using the most natural primitives for its job.

After the SVG closes, the HTML host resumes (footer paragraph) —
no manual unwinding needed.

In [9]:
class Section5(HtmlBuilderHandler):
    def main(self, root):
        body = root.body(font_family='sans-serif', color='#2c3e50')
        card = body.div(
            background='white', padding='16px',
            border='1px solid #bdc3c7', border_radius=8,
            width='360px',
        )
        card.h3('Q1 sales', margin='0 0 8px 0')
        chart = card.svg(width=320, height=140, viewBox='0 0 320 140')
        bars = [('Jan', 80, '#3498db'), ('Feb', 60, '#9b59b6'),
                ('Mar', 100, '#e67e22'), ('Apr', 45, '#1abc9c')]
        x = 10
        for label, h, color in bars:
            chart.rect(x=x, y=120 - h, width=60, height=h, fill=color)
            chart.text(label, x=x + 30, y=135,
                       font_size=11, text_anchor='middle', fill='#2c3e50')
            x += 80
        cap = chart.html(x=200, y=4, width=110, height=22)
        cap.div(
            '↑ Apr below target',
            color='#e74c3c', font_size='11px', font_weight='600',
            text_align='right',
        )
        card.p(
            'Source: internal forecast',
            font_size='11px', color='#7f8c8d', margin='8px 0 0 0',
        )


p5 = Section5(); p5.create(); p5.build()
print(p5.render(pretty=True))

<body style="font-family: sans-serif; color: #2c3e50">
  <div style="background: white; padding: 16px; border: 1px solid #bdc3c7; border-radius: 8; width: 360px">
    <h3 style="margin: 0 0 8px 0">Q1 sales</h3>
<svg width="320" height="140" viewBox="0 0 320 140"><rect x="10" y="40" width="60" height="80" fill="#3498db" /><text x="40" y="135" font-size="11" text-anchor="middle" fill="#2c3e50">Jan</text><rect x="90" y="60" width="60" height="60" fill="#9b59b6" /><text x="120" y="135" font-size="11" text-anchor="middle" fill="#2c3e50">Feb</text><rect x="170" y="20" width="60" height="100" fill="#e67e22" /><text x="200" y="135" font-size="11" text-anchor="middle" fill="#2c3e50">Mar</text><rect x="250" y="75" width="60" height="45" fill="#1abc9c" /><text x="280" y="135" font-size="11" text-anchor="middle" fill="#2c3e50">Apr</text><foreignObject xmlns="http://www.w3.org/1999/xhtml" x="200" y="4" width="110" height="22"><div style="color: #e74c3c; font-size: 11px; font-weight: 600; text-align

In [10]:
HTML(p5.render())

## 6. Inspection — how the framework realises sub-builders

The whole mechanism rests on two pieces of state:

1. **Schema-level declaration**: when an `@subbuilder("svg")` is
   declared (in `Html5Extensions.svg`), the class schema entry
   for `svg` carries `is_subbuilder=True` and
   `subbuilder_name='svg'`. The dispatch reads it at attach time
   to know it should switch builder.
2. **Per-node state**: once attached, the `<svg>` node carries an
   `SvgBuilder` instance on its `_builder` slot. Any subsequent
   call (`s.rect()`, `s.html()`) resolves the builder through
   `node._resolve_builder()` and dispatches through the new
   grammar.

The cell below shows both, on the live `Section1` page from §1.

In [11]:
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.contrib.svg import SvgBuilder

# Schema-level: the declaration is class-wide.
info = HtmlBuilder()._get_schema_info('svg')
print('schema info for svg:', dict(info))

# Per-node: after create() runs, the svg node carries the sub-builder
# on its _builder slot. Below we walk the source down to the <svg>
# node of Section1 and inspect its active dialect.
body_node = next(iter(p1.source))
svg_node = next(c for c in body_node.value if c.node_tag == 'svg')
active = svg_node._resolve_builder()
print(f'svg node active builder type: {type(active).__name__}')
print(f'svg node active builder _name: {active._name!r}')
print(f'svg node active builder is SvgBuilder instance: {isinstance(active, SvgBuilder)}')

schema info for svg: {'handler_name': '_subb_svg', 'is_subbuilder': True, 'subbuilder_name': 'svg'}
svg node active builder type: SvgBuilder
svg node active builder _name: 'svg'
svg node active builder is SvgBuilder instance: True
